# 🚀 Your First AI Agent: From Prompt to Action

**Welcome to the Kaggle 5-day Agents course!**

This notebook is your first step into building AI agents. An agent can do more than just respond to a prompt — it can **take actions** to find information or get things done.

In this notebook, you'll:

- ✅ Install [Agent Development Kit (ADK)](https://google.github.io/adk-docs/)
- ✅ Configure your API key to use the Gemini model
- ✅ Build your first simple agent
- ✅ Run your agent and watch it use a tool (like Google Search) to answer a question


## ⚙️ Section 1: Setup

### 1.1: Install dependencies

The Kaggle Notebooks environment includes a pre-installed version of the [google-adk](https://google.github.io/adk-docs/) library for Python and its required dependencies, so you don't need to install additional packages in this notebook.

To install and use ADK in your own Python development environment outside of this course, you can do so by running:

```
pip install google-adk
```

### 1.2: Configure your Gemini API Key

This notebook uses the [Gemini API](https://ai.google.dev/gemini-api/docs), which requires authentication.

**1. Get your API key**

If you don't have one already, create an [API key in Google AI Studio](https://aistudio.google.com/app/api-keys).

**2. Add the key to Kaggle Secrets**

Next, you will need to add your API key to your Kaggle Notebook as a Kaggle User Secret.

1. In the top menu bar of the notebook editor, select `Add-ons` then `Secrets`.
2. Create a new secret with the label `GOOGLE_API_KEY`.
3. Paste your API key into the "Value" field and click "Save".
4. Ensure that the checkbox next to `GOOGLE_API_KEY` is selected so that the secret is attached to the notebook.

**3. Authenticate in the notebook**

Run the cell below to complete authentication.

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6LEcpI6U61-7mIpLny0M_JmZxYsTDZ8EOWre7JqPKqGeg" 

In [15]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if GOOGLE_API_KEY is not None:
    print("✅ Gemini API key setup complete.")
else:
    raise ValueError("GOOGLE_API_KEY is not set. Set it as an environment variable first.")

✅ Gemini API key setup complete.


### 1.3: Import ADK components

Now, import the specific components you'll need from the Agent Development Kit and the Generative AI library. This keeps your code organized and ensures we have access to the necessary building blocks.

In [16]:
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


### 1.4: Helper functions

We'll define some helper functions. If you are running this outside the Kaggle environment, you don't need to do this.

In [19]:
# Define helper functions that will be reused throughout the notebook

from IPython.display import display, HTML
import importlib


def _get_list_running_servers():
    try:
        serverapp = importlib.import_module("jupyter_server.serverapp")
    except Exception:
        return None

    return getattr(serverapp, "list_running_servers", None)


def get_adk_proxy_url():
    ADK_PORT = "8000"
    list_running_servers = _get_list_running_servers()
    is_kaggle = os.getenv("KAGGLE_KERNEL_RUN_TYPE") is not None and list_running_servers is not None

    if is_kaggle:
        PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
        servers = list(list_running_servers())
        if not servers:
            raise Exception("No running Jupyter servers found.")

        base_url = servers[0]["base_url"]
        try:
            path_parts = base_url.split("/")
            kernel = path_parts[2]
            token = path_parts[3]
        except IndexError:
            raise Exception(f"Could not parse kernel/token from base URL: {base_url}")

        url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
        ui_url = f"{PROXY_HOST}{url_prefix}"
        banner_text = (
            "The ADK web UI is not running yet. In Kaggle, run the next cell first, then come back "
            "and open the link below."
        )
        action_text = "Open ADK Web UI (after running the cell below) ↗"
    else:
        url_prefix = "/"
        ui_url = f"http://127.0.0.1:{ADK_PORT}"
        banner_text = (
            "The ADK web UI is not running yet. On a local notebook, run the next cell and then "
            "open http://127.0.0.1:8000 in your browser."
        )
        action_text = "Open ADK Web UI at localhost ↗"

    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            {banner_text}
        </div>
        <a href='{ui_url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            {action_text}
        </a>
    </div>
    """

    display(HTML(styled_html))

    return url_prefix


print("✅ Helper functions defined.")

✅ Helper functions defined.


### 1.5: Configure Retry Options

When working with LLMs, you may encounter transient errors like rate limits or temporary service unavailability. Retry options automatically handle these failures by retrying the request with exponential backoff.

In [20]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)

---

## 🤖 Section 2: Your first AI Agent with ADK

### 🤔 2.1 What is an AI Agent?

You've probably used an LLM like Gemini before, where you give it a prompt and it gives you a text response.

`Prompt -> LLM -> Text`

An AI Agent takes this one step further. An agent can think, take actions, and observe the results of those actions to give you a better answer.

`Prompt -> Agent -> Thought -> Action -> Observation -> Final Answer`

In this notebook, we'll build an agent that can take the action of searching Google. Let's see the difference!

### 2.2 Define your agent

Now, let's build our agent. We'll configure an `Agent` by setting its key properties, which tell it what to do and how to operate.

To learn more, check out the documentation related to [agents in ADK](https://google.github.io/adk-docs/agents/).

These are the main properties we'll set:

- **name** and **description**: A simple name and description to identify our agent.
- **model**: The specific LLM that will power the agent's reasoning. We'll use "gemini-2.5-flash-lite".
- **instruction**: The agent's guiding prompt. This tells the agent what its goal is and how to behave.
- **tools**: A list of [tools](https://google.github.io/adk-docs/tools/) that the agent can use. To start, we'll give it the `google_search` tool, which lets it find up-to-date information online.

In [21]:
root_agent = Agent(
    name="helpful_assistant",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    description="A simple agent that can answer general questions.",
    instruction="You are a helpful assistant. Use Google Search for current info or if unsure.",
    tools=[google_search],
)

print("✅ Root Agent defined.")

✅ Root Agent defined.


### 2.3 Run your agent

Now it's time to bring your agent to life and send it a query. To do this, you need a [`Runner`](https://google.github.io/adk-docs/runtime/), which is the central component within ADK that acts as the orchestrator. It manages the conversation, sends our messages to the agent, and handles its responses.

**a. Create an `InMemoryRunner` and tell it to use our `root_agent`:**

In [22]:
runner = InMemoryRunner(agent=root_agent)
print("✅ Runner created.")

✅ Runner created.


👉 Note that we are using the Python Runner directly in this notebook. You can also run agents using ADK command-line tools such as `adk run`, `adk web`, or `adk api_server`. To learn more, check out the documentation related to [runtime in ADK](https://google.github.io/adk-docs/runtime/).

**b. Now you can call the `.run_debug()` method to send our prompt and get an answer.**

👉 This method abstracts the process of session creation and maintenance and is used in prototyping. We'll explore "what sessions are and how to create them" on Day 3.



The Gemini API might be UNAVAILABLE due to high demand, not because of a notebook syntax or ADK setup issue. The issue is external and temporary; re-running the cell later or switching to another Gemini model is the most likely fix.

In [24]:
response = await runner.run_debug(
    "What is Agent Development Kit from Google? What languages is the SDK available in?"
)

helpful_assistant > The Google Agent Development Kit (ADK) is an open-source framework designed to simplify the creation, debugging, and deployment of AI agents and multi-agent systems at an enterprise scale. It allows developers to build agents that can range from personal AI assistants to complex business workflows. The ADK applies software development principles to AI agent creation, offering precise control over agent behavior and orchestration, and a rich tool ecosystem for integrating third-party applications and custom code. It is optimized for Google Cloud and Gemini models but is model-agnostic and deployment-agnostic. The ADK also features a built-in development UI for testing, evaluating, and debugging agents.

The Agent Development Kit (SDK) is available in the following programming languages:
*   Python
*   TypeScript
*   Go
*   Java
*   Kotlin


You can see a summary of ADK and its available languages in the response.

### 2.4 How does it work?

The agent performed a Google Search to get the latest information about ADK, and it knew to use this tool because:

1. The agent inspects and is aware of which tools it has available to use.
2. The agent's instructions specify the use of the search tool to get current information or if it is unsure of an answer.

The best way to see the full, detailed trace of the agent's thoughts and actions is in the **ADK web UI**, which we'll set up later in this notebook.

And we'll cover more detailed workflows for logging and observability later in the course.

### 🚀 2.5 Your Turn!

This is your chance to see the agent in action. Ask it a question that requires current information.

Try one of these, or make up your own:

- What's the weather in London?
- Who won the last soccer world cup?
- What new movies are showing in theaters now?

In [25]:
response = await runner.run_debug("What was the result of the England vs. DRC game in world cup 2026?")

helpful_assistant > The result of the England vs. DRC game in the World Cup 2026 was England 2-1 DR Congo. Harry Kane scored twice in the second half to secure the win for England after DR Congo had taken an early lead. With this victory, England advanced to the Round of 16, where they are scheduled to play Mexico.


---

## 💻 Section 3: Try the ADK Web Interface

### Overview

ADK includes a built-in web interface for interactively chatting with, testing, and debugging your agents.

<img width="1200" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/adk-web-ui.gif" alt="ADK Web UI" />

To use the ADK web UI, you'll need to create an agent with Python files using the `adk create` command.

Run the command below to generate a `sample-agent` folder that contains all the necessary files, including `agent.py` for your code, an `.env` file with your API key pre-configured, and an `__init__.py` file:

In [26]:
!adk create sample-agent --model gemini-2.5-flash-lite --api_key $GOOGLE_API_KEY


Agent created in /Users/ashkanm/Downloads/Python_Projects/Current_Projects_MLOps/AI-Agents-5Day/AI-Agents-Intensive-Course-with-Google-5Day/Day 1/notebooks/sample-agent:
- .env
- .gitignore
- __init__.py
- agent.py

⚠️  WARNING: Secrets (like GOOGLE_API_KEY) are stored in .env.



Get your ADK web UI URL for your current environment:

In [27]:
url_prefix = get_adk_proxy_url()

Now we can run ADK web:

In [28]:
import os

if os.getenv("KAGGLE_KERNEL_RUN_TYPE"):
    !adk web --url_prefix {url_prefix}
else:
    !adk web

2026-07-01 20:58:31,994 - INFO - service_factory.py:266 - Using in-memory memory service
2026-07-01 20:58:31,994 - INFO - local_storage.py:88 - Using per-agent session storage rooted at /Users/ashkanm/Downloads/Python_Projects/Current_Projects_MLOps/AI-Agents-5Day/AI-Agents-Intensive-Course-with-Google-5Day/Day 1/notebooks
2026-07-01 20:58:31,994 - INFO - local_storage.py:120 - Using per-agent artifact storage rooted at /Users/ashkanm/Downloads/Python_Projects/Current_Projects_MLOps/AI-Agents-5Day/AI-Agents-Intensive-Course-with-Google-5Day/Day 1/notebooks
/opt/anaconda3/envs/temp/lib/python3.13/site-packages/google/adk/cli/fast_api.py:553: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/opt/anaconda3/envs/temp/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_creden

Now you can access the ADK dev UI using the link above.

Once you open the link, you'll see the ADK web interface where you can ask your ADK agent questions.

Note: This sample agent does not have any tools enabled (like Google Search). It is a basic agent designed specifically to let you explore the UI features.

If you are running this on Kaggle, do not share the proxy link with anyone because it contains an authentication token in the URL.

---

## ✅ Congratulations!

You've built and run your first agent with ADK! You've just seen the core concept of agent development in action.

The big takeaway is that your agent didn't just *respond*—it **reasoned** that it needed more information and then **acted** by using a tool. This ability to take action is the foundation of all agent-based AI.


### 📚 Learn More

Refer to the following documentation to learn more:

- [ADK Documentation](https://google.github.io/adk-docs/)
- [ADK Quickstart for Python](https://google.github.io/adk-docs/get-started/python/)
- [ADK Agents Overview](https://google.github.io/adk-docs/agents/)
- [ADK Tools Overview](https://google.github.io/adk-docs/tools/)

### 🎯 Next Steps

Ready for the next challenge? Continue to the next notebook to learn how to **architect multi-agent systems.**